In [ ]:
"""
Script 8: Statistical Summary and Report
Generate a comprehensive statistical summary report
"""

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

def generate_summary_report():
    """Generate a comprehensive statistical summary report"""
    
    print("\n" + "="*80)
    print("COMPREHENSIVE STATISTICAL ANALYSIS REPORT")
    print("="*80)
    print(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    report_lines = []
    report_lines.append("="*80)
    report_lines.append("COMPREHENSIVE STATISTICAL ANALYSIS REPORT")
    report_lines.append("="*80)
    report_lines.append(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Load all analysis results
    try:
        # Load data
        df = pd.read_csv('outputs/cleaned_data.csv')
        df['Date'] = pd.to_datetime(df['Date'])
        
        # 1. Dataset Overview
        report_lines.append("\n1. DATASET OVERVIEW")
        report_lines.append("─" * 80)
        report_lines.append(f"Total Records: {len(df)}")
        report_lines.append(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
        report_lines.append(f"Duration: {(df['Date'].max() - df['Date'].min()).days} days")
        report_lines.append(f"Number of Variables: {df.shape[1] - 1}")
        report_lines.append("")
        
        # 2. Descriptive Statistics
        try:
            desc_stats = pd.read_csv('outputs/descriptive_statistics.csv')
            report_lines.append("\n2. DESCRIPTIVE STATISTICS SUMMARY")
            report_lines.append("─" * 80)
            report_lines.append(desc_stats.to_string(index=False))
            report_lines.append("")
        except FileNotFoundError:
            report_lines.append("\n2. DESCRIPTIVE STATISTICS")
            report_lines.append("─" * 80)
            report_lines.append("(Run 02_descriptive_stats.py to generate)")
            report_lines.append("")
        
        # 3. Price Analysis
        report_lines.append("\n3. PRICE ANALYSIS")
        report_lines.append("─" * 80)
        price = df['Close Price']
        report_lines.append(f"Minimum Price: {price.min():.2f}")
        report_lines.append(f"Maximum Price: {price.max():.2f}")
        report_lines.append(f"Price Range: {price.max() - price.min():.2f}")
        report_lines.append(f"Average Price: {price.mean():.2f}")
        report_lines.append(f"Median Price: {price.median():.2f}")
        
        # Calculate returns
        total_return = ((price.iloc[-1] - price.iloc[0]) / price.iloc[0]) * 100
        report_lines.append(f"Total Return: {total_return:.2f}%")
        report_lines.append(f"Daily Volatility: {price.pct_change().std() * 100:.2f}%")
        report_lines.append("")
        
        # 4. Normality Tests
        try:
            normality_tests = pd.read_csv('outputs/normality_tests.csv')
            report_lines.append("\n4. NORMALITY TESTS SUMMARY")
            report_lines.append("─" * 80)
            
            for column in normality_tests['Column'].unique():
                col_data = normality_tests[normality_tests['Column'] == column]
                normal_count = (col_data['Interpretation'] == 'Possibly Normal').sum()
                total_tests = len(col_data)
                report_lines.append(f"{column}: {normal_count}/{total_tests} tests indicate normality")
                
                if normal_count >= total_tests * 0.75:
                    report_lines.append(f"  → Likely NORMAL distribution")
                else:
                    report_lines.append(f"  → Likely NON-NORMAL distribution")
            report_lines.append("")
        except FileNotFoundError:
            report_lines.append("\n4. NORMALITY TESTS")
            report_lines.append("─" * 80)
            report_lines.append("(Run 05_normality_tests.py to generate)")
            report_lines.append("")
        
        # 5. Correlation Analysis
        try:
            pearson_corr = pd.read_csv('outputs/pearson_correlation.csv', index_col=0)
            report_lines.append("\n5. CORRELATION ANALYSIS (PEARSON)")
            report_lines.append("─" * 80)
            report_lines.append(pearson_corr.to_string())
            report_lines.append("")
        except FileNotFoundError:
            report_lines.append("\n5. CORRELATION ANALYSIS")
            report_lines.append("─" * 80)
            report_lines.append("(Run 06_correlation_analysis.py to generate)")
            report_lines.append("")
        
        # 6. Hypothesis Testing
        try:
            hyp_tests = pd.read_csv('outputs/hypothesis_tests.csv')
            report_lines.append("\n6. HYPOTHESIS TESTING RESULTS")
            report_lines.append("─" * 80)
            for idx, row in hyp_tests.iterrows():
                report_lines.append(f"Test {idx+1}: {row['Test']}")
                report_lines.append(f"  Conclusion: {row['Conclusion']}")
            report_lines.append("")
        except FileNotFoundError:
            report_lines.append("\n6. HYPOTHESIS TESTING")
            report_lines.append("─" * 80)
            report_lines.append("(Run 07_hypothesis_testing.py to generate)")
            report_lines.append("")
        
        # 7. Key Findings
        report_lines.append("\n7. KEY FINDINGS AND CONCLUSIONS")
        report_lines.append("─" * 80)
        
        # Trend analysis
        if total_return > 0:
            report_lines.append(f"✓ Overall UPTREND: Price increased by {total_return:.2f}%")
        else:
            report_lines.append(f"✗ Overall DOWNTREND: Price decreased by {abs(total_return):.2f}%")
        
        # Volatility assessment
        daily_vol = price.pct_change().std() * 100
        report_lines.append(f"✓ Volatility Level: {daily_vol:.2f}% (Daily)")
        
        # Moving averages
        if df['3 Day MV.'].iloc[-1] > df['5 Day MV.'].iloc[-1]:
            report_lines.append("✓ Short-term signal: 3-Day MA above 5-Day MA (Bullish)")
        else:
            report_lines.append("✗ Short-term signal: 3-Day MA below 5-Day MA (Bearish)")
        
        report_lines.append("")
        
        # 8. Recommendations
        report_lines.append("\n8. RECOMMENDATIONS")
        report_lines.append("─" * 80)
        report_lines.append("• Monitor daily volatility trends for risk assessment")
        report_lines.append("• Use moving averages for trend confirmation")
        report_lines.append("• Consider non-parametric tests for non-normal distributions")
        report_lines.append("• Track correlation changes over different time periods")
        report_lines.append("")
        
        # Print and save
        report_text = "\n".join(report_lines)
        print(report_text)
        
        # Save to file
        Path('outputs').mkdir(exist_ok=True)
        with open('outputs/statistical_summary_report.txt', 'w') as f:
            f.write(report_text)
        
        print(f"\n✓ Report saved to outputs/statistical_summary_report.txt")
        
    except Exception as e:
        print(f"Error generating report: {e}")

def generate_results_index():
    """Generate an index of all output files"""
    print("\n" + "="*80)
    print("OUTPUT FILES INDEX")
    print("="*80 + "\n")
    
    output_dir = Path('outputs')
    
    if not output_dir.exists():
        print("No outputs directory found. Run analysis scripts first.")
        return
    
    print("Generated Files:\n")
    
    # CSV files
    csv_files = list(output_dir.glob('*.csv'))
    if csv_files:
        print("Data Files:")
        for f in sorted(csv_files):
            print(f"  • {f.name}")
        print()
    
    # Text files
    txt_files = list(output_dir.glob('*.txt'))
    if txt_files:
        print("Reports:")
        for f in sorted(txt_files):
            print(f"  • {f.name}")
        print()
    
    # Plot files
    plot_dir = output_dir / 'plots'
    if plot_dir.exists():
        plot_files = list(plot_dir.glob('*.png'))
        if plot_files:
            print("Visualizations:")
            for f in sorted(plot_files):
                print(f"  • {f.name}")
            print()

def create_quick_reference():
    """Create a quick reference guide"""
    print("\n" + "="*80)
    print("QUICK REFERENCE GUIDE")
    print("="*80 + "\n")
    
    guide = """
RUNNING THE ANALYSIS PIPELINE:
────────────────────────────────────────────────────────────────────────────────

Step 1: Load and Explore Data
  $ python 01_data_loading.py
  Outputs: cleaned_data.csv

Step 2: Calculate Descriptive Statistics
  $ python 02_descriptive_stats.py
  Outputs: descriptive_statistics.csv

Step 3: Create Visualizations
  $ python 03_data_visualization.py
  Outputs: Multiple PNG files in plots/

Step 4: Perform Trend Analysis
  $ python 04_trend_analysis.py
  Outputs: data_with_indicators.csv

Step 5: Test for Normality
  $ python 05_normality_tests.py
  Outputs: normality_tests.csv

Step 6: Analyze Correlations
  $ python 06_correlation_analysis.py
  Outputs: pearson_correlation.csv, spearman_correlation.csv

Step 7: Perform Hypothesis Tests
  $ python 07_hypothesis_testing.py
  Outputs: hypothesis_tests.csv

Step 8: Generate Summary Report
  $ python 08_statistical_summary.py
  Outputs: statistical_summary_report.txt

────────────────────────────────────────────────────────────────────────────────
All outputs are saved in the 'outputs/' directory
Visualizations are saved in 'outputs/plots/' subdirectory
"""
    
    print(guide)

if __name__ == "__main__":
    generate_summary_report()
    generate_results_index()
    create_quick_reference()
    print("\n✓ Statistical summary completed!")